# Milestone 4 Lab, notebook 1 of 7: A figure, traced by hand

## Where we are

| Notebook | What you do |
|---|---|
| **1. A figure, traced by hand** (you are here) | Find every record behind one figure yourself, and add them up |
| 2. Run records | Give every pipeline run an id and a record, and keep every run |
| 3. Lineage | Save which records made each figure, and prove it adds up |
| 4. Two runs compared | Find what changed between two runs: settings, inputs, figures, records |
| 5. A trace of the analyst | Give the analyst the new tools, and record every step it takes |
| 6. Scoring an agent | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| 7. Two experiments | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **0**. A full run of this lab costs about 5 US cents.

## What this notebook is for

After milestone 3 the analyst gives a figure, a warning, and a confidence level. Two questions still have no good answer:

1. **Where did this number come from?** The analyst shows five sample records. A sample is not proof.
2. **Why did this number change?** Nobody wrote it down.

In this notebook you answer the first question yourself, for one figure, with no AI. It works, and it is slow. That is the point: the rest of the milestone makes the system do it.

> If you have not done milestones 1 to 3, do them first.

In [ ]:
import pandas as pd
from lab_helpers import load_result, load_source_tables, save_result, show_doc

## Step 1: What the business is asking for now

Here is this milestone's requirement, from the **BRD**.

In [ ]:
show_doc("BRD.md", "1. Business problem")

## Step 2: One figure, two values

This is the answer key from milestone 3. Look at the first line: the trail water bottle made **$5,156.94** in October.

In [ ]:
answer_key = load_result("answer_key")

pd.DataFrame(answer_key["figures"])[["what", "entity_id", "kind", "value"]]

Now the answer key from milestone 2, the one before. The same line says **$5,188.20**.

In [ ]:
old_key = load_result("answer_key_milestone_2")

old_figure = old_key["figures"][0]
new_figure = answer_key["figures"][0]

print("Before:", old_figure["what"], old_figure["value"])
print("Now:   ", new_figure["what"], new_figure["value"])

The same product, the same month, two different numbers. You happen to know why, because you built milestone 3. A new person on the team would not. The system itself has no record of it.

Keep that in mind. First, the other question: where does $5,156.94 come from?

## Step 3: Load the records

In [ ]:
tables = load_source_tables()

for name, table in tables.items():
    print(name, len(table), "rows")

## Step 4: Find the order lines

A figure belongs to one product and one month. We want the trail water bottle (`P001`) in October 2026.

An order line has no date. Its order has. So we join each line to its order, then keep the lines we want. Cancelled orders do not count.

In [ ]:
PRODUCT = "P001"
MONTH = "2026-10"

lines = tables["order_items"].merge(tables["orders"], on="order_id")
lines["month"] = lines["order_date"].dt.strftime("%Y-%m")

is_ours = (lines["product_id"] == PRODUCT) & (lines["month"] == MONTH)
not_cancelled = lines["status"] != "cancelled"
our_lines = lines[is_ours & not_cancelled]

print(len(our_lines), "order lines")

Milestone 3 taught us that some lines appear twice, and that the extra copies are set aside. `drop_duplicates` keeps the first copy of each.

In [ ]:
kept_lines = our_lines.drop_duplicates()

print(len(our_lines), "order lines, of which", len(kept_lines), "are kept")

Three terms come from order lines: units sold, gross sales, and discounts.

In [ ]:
units_sold = kept_lines["quantity"].sum()
gross_sales = (kept_lines["quantity"] * kept_lines["unit_list_price"]).sum()
discounts = kept_lines["discount_amount"].sum()

print("Units sold: ", units_sold)
print("Gross sales:", gross_sales)
print("Discounts:  ", discounts)

## Step 5: Find the refunds, shipments, fees, and cost

Each of these tables has its own date, and each counts in the month of that date. We do the same thing four times.

In [ ]:
refunds = tables["refunds"]
refunds = refunds[refunds["product_id"] == PRODUCT]
refunds = refunds[refunds["refund_date"].dt.strftime("%Y-%m") == MONTH]
refunds = refunds.drop_duplicates()

refund_total = refunds["refund_amount"].sum()
return_cost = refunds["return_handling_cost"].sum()

print(len(refunds), "refunds:", refund_total, "refunded,", return_cost, "return cost")

In [ ]:
shipments = tables["shipments"]
shipments = shipments[shipments["product_id"] == PRODUCT]
shipments = shipments[shipments["ship_date"].dt.strftime("%Y-%m") == MONTH]
shipments = shipments.drop_duplicates()

fulfillment_cost = shipments["pick_pack_cost"].sum()
shipping_cost = shipments["carrier_cost"].sum()

print(len(shipments), "shipments:", fulfillment_cost, "fulfilment,", shipping_cost, "shipping")

In [ ]:
fees = tables["payment_fees"]
fees = fees[fees["product_id"] == PRODUCT]
fees = fees[fees["capture_date"].dt.strftime("%Y-%m") == MONTH]
fees = fees.drop_duplicates()

payment_fees = round(fees["fee_amount"].sum(), 2)

print(len(fees), "payment fees:", payment_fees)

In [ ]:
costs = tables["product_costs"]
costs = costs[(costs["product_id"] == PRODUCT) & (costs["month"] == MONTH)]

unit_cost = costs["unit_cost"].iloc[0]
product_cost = unit_cost * units_sold

print(len(costs), "cost record:", unit_cost, "a unit, so", product_cost, "for", units_sold, "units")

## Step 6: Add it all up

The three metrics are built from those terms, as defined in milestone 1.

In [ ]:
net_sales = gross_sales - discounts - refund_total
gross_profit = net_sales - product_cost
contribution_profit = gross_profit - fulfillment_cost - shipping_cost - payment_fees - return_cost
contribution_profit = round(float(contribution_profit), 2)

print("Net sales:          ", net_sales)
print("Gross profit:       ", gross_profit)
print("Contribution profit:", contribution_profit)
print("The answer key says:", new_figure["value"])

They match. You have traced the figure: it is built from exactly these records.

In [ ]:
record_counts = {
    "order_items": len(kept_lines),
    "refunds": len(refunds),
    "shipments": len(shipments),
    "payment_fees": len(fees),
    "product_costs": len(costs),
}

print(record_counts)
print("Records behind this one figure:", sum(record_counts.values()))

## Step 7: And the old figure?

One guess: before milestone 3 the extra copies were counted. Let us test the guess. Do the same sums again, this time **without** dropping the copies.

In [ ]:
def contribution_profit_from(lines, refunds, shipments, fees):
    """The same sums as steps 4 to 6, for whatever records are passed in."""
    units = lines["quantity"].sum()
    gross = (lines["quantity"] * lines["unit_list_price"]).sum()
    net = gross - lines["discount_amount"].sum() - refunds["refund_amount"].sum()
    profit = net - unit_cost * units
    profit = profit - shipments["pick_pack_cost"].sum() - shipments["carrier_cost"].sum()
    profit = profit - fees["fee_amount"].sum() - refunds["return_handling_cost"].sum()
    return round(float(profit), 2)


all_shipments = tables["shipments"]
all_shipments = all_shipments[all_shipments["product_id"] == PRODUCT]
all_shipments = all_shipments[all_shipments["ship_date"].dt.strftime("%Y-%m") == MONTH]

all_fees = tables["payment_fees"]
all_fees = all_fees[all_fees["product_id"] == PRODUCT]
all_fees = all_fees[all_fees["capture_date"].dt.strftime("%Y-%m") == MONTH]

with_copies = contribution_profit_from(our_lines, refunds, all_shipments, all_fees)

print("Counting the copies:", with_copies)
print("The old key says:   ", old_figure["value"])

The guess was right: counting the copies gives the old figure, $5,188.20.

But notice what we just did. We **guessed** a reason and tested it. That worked because we remembered milestone 3. A system should not depend on anyone's memory. It should have recorded both calculations, so that the difference can be looked up.

## Step 8: Save what you found

This is the answer key for this milestone's new questions. Every later notebook is checked against it.

In [ ]:
lineage_key = {
    "product_id": PRODUCT,
    "month": MONTH,
    "contribution_profit": contribution_profit,
    "record_counts": record_counts,
    "total_records": sum(record_counts.values()),
    "gross_sales": float(gross_sales),
    "earlier_contribution_profit": with_copies,
    "change": round(contribution_profit - with_copies, 2),
}

save_result("answer_key_lineage", lineage_key)
lineage_key

## What we now know

- One figure, the trail water bottle's October contribution profit, is built from **754 records** in five tables.
- Tracing it by hand took this whole notebook. The profit table has 24 rows and each row has three metrics.
- The old figure, $5,188.20, comes from counting three duplicated order lines and their shipments and fees. We found that by guessing, because **nothing recorded it**.
- So the system needs two things: a saved link from every figure to its records (**lineage**), and a kept record of every calculation (**run history**).

**Next:** notebook 2, where every calculation gets an id and a record.